<a href="https://colab.research.google.com/github/JCARNEIROX/IA367-Aprendizado-Reforco/blob/main/Lista2_Ex12_RA239738_RA256389.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IA368FF - Aprendizado por Reforço**  
1º Semestre de 2024  
Prof. Denis Fantinato  

In [3]:
from copy import deepcopy
import numpy as np

from collections import defaultdict

# Criando o MDP

**Grid World 4x3**  
Vamos criar um Grid World 4x3 como um MDP.  

O MDP é definido por:  
                        MDP = (𝑆, 𝐴, 𝑅, ℙ, 𝛾)
com
- 𝑆: conjunto de possíveis estados.
- 𝐴: conjunto de ações.
- 𝑅 ∶ 𝑆 → ℝ: mapa de recompensa para cada estado.
- ℙ: probabilidade de transição de um estado para outro dada uma ação.
- 𝛾: fator de desconto. Um número entre 0 e 1.  
  
Neste mesmo bloco, definimos as probabilidades de transição de estados. Note que o agente tem 80\% de chance de seguir na direção da ação escolhida e 10\% de chance para cada direção perpendicular.

In [4]:
def createMDP():

    '''
    definição do ambiente
    '''

    S       = [(i,j) for i in range(1,5)
                     for j in range(1,4) if (i,j) != (2,2)]

    goals   = [(4,3), (4,2)]
    actions = ["UP", "DOWN", "LEFT", "RIGHT"]
    A       = {s : actions
               for s in S }

    R        = {s : -0.04 for s in S}
    R[(4,3)] =  1
    R[(4,2)] = -1

    P        = { (s,a) : pvals(s, a, S) for s in S for a in A[s] }

    gamma    = .9

    return (S,A,R,P,gamma)


def move(s, a, S):
    i, j = s
    if a == "UP":
        sp = (i, j+1)
    elif a == "DOWN":
        sp = (i, j-1)
    elif a == "LEFT":
        sp = (i-1, j)
    elif a == "RIGHT":
        sp = (i+1, j)
    elif a is None:
        return s

    if sp in S:
        return sp

    return s

def succ(a):
    return {"UP": "RIGHT", "DOWN": "LEFT", "RIGHT": "DOWN", "LEFT": "UP", None : None}[a]

def pred(a):
    return {"UP": "LEFT", "DOWN": "RIGHT", "RIGHT": "UP", "LEFT": "DOWN", None : None}[a]

def pvals(s, a, S):
    return [(0.8, move(s, a, S)), (0.1, move(s, succ(a), S)), (0.1, move(s, pred(a), S))]

def policyIteration(mdp):

    S, A, R, P, gamma = mdp

    U       = { s : 0.0 for s in S }
    # política inicial executa a primeira ação da lista
    pi      = { s : A[s][0] for s in S }
    changed = True

    while changed:
        U       = policyEvaluation(pi, U, mdp)
        changed = False
        for s in S:
            # maior valor esperado utilizando a matriz utilidade atual
            maxU = max(expVal(P[(s,a)],U) for a in A[s])
            # maior valor esperado utilizando a política atual
            piU  = expVal(P[(s,pi[s])],U)

            # Se a utilidade ganhar, atualiza a política
            if maxU > piU:
                idx     = np.argmax( [expVal(P[(s,a)],U) for a in A[s]] )
                pi[s]   = A[s][idx]
                changed = True
                #print(s, maxU, piU)
    return pi

def policyEvaluation(pi, U, mdp):
    S, A, R, P, gamma = mdp
    for s in S:
        U[s] = R[s] + gamma * expVal(P[(s, pi[s])],U)
    return U

def expVal(ps, U):
    '''
    retorna o valor esperado da utilidade dadas as probabilidades
    de possíveis estados consequentes armazenados em ps.
    '''
    return sum(p*U[s] for p, s in ps)

Para confirmar se entendeu esse bloco:    
- Onde a recompensa pode ser ajustada?

  `R:` A recompensa pode ser ajustada trocando o valor "-0.04" na linha de código 15.
- Como ficaria a lista `S`? Escreva os estados na ordem correta.

  `R:` S = [(1, 1),(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2), (3, 3), (4, 1), (4, 2), (4, 3)]

- O que há em:
  - `A[(3,2)]`?

    `R:` A[(3,2)] = ['UP', 'DOWN', 'LEFT', 'RIGHT']

  - `A[(2,2)]`?

    `R:` A[(2,2)] = State (2,2) does not exist

  - `A[(4,2)]`?

    `R:` A[(4,2)] = ['UP', 'DOWN', 'LEFT', 'RIGHT']
- Qual a saída para `P[((3,3),"RIGHT")] = pvals((3,3), "RIGHT", S)`?

    `R:`P[((3,3),"RIGHT")] = [(0.8, (4, 3)), (0.1, (3, 2)), (0.1, (3, 3))]    
- O que acontece se o resultado for um estado fora do grid?

    `R:` O Agente não se movimenta e permanece no estado atual.


# Algoritmo Estimativa Direta

Aprendizado Passivo.  
É necessário obter as sequências-amostras e depois estimar as utilidades

In [5]:
def directEst(model, s, goals, R, gamma, nextState, maxLen=100):
    trial = [(s, R[s])]
    while s not in goals:
        s = nextState(s)
        trial.append( (s, R[s]) )
        if len(trial) > maxLen:
            return None
    for i, (s, r) in enumerate(trial):
        u        = sum(r*(gamma**j)
                        for j, (si, r) in enumerate(trial[i:]))
        model[s] = (model[s][0] + u, model[s][1] + 1)
    return model

def runDirectEst(mdp, pi, nTrials):
    S, A, R, P, gamma = mdp

    model = defaultdict(lambda: (0.0, 0))
    s0    = (1,1)
    goals = [(4,3), (4,2)]

    for trials in range(nTrials):
        model = directEst(model, s0, goals, R, gamma, performAction(pi, P))
        if model is None:
            break

    return model

# Escolhe próximo estado dado uma ação
def performAction(pi, P):
    def nextState(s):
        ps     = P[(s, pi[s])]
        probs  = list(map(lambda x: x[0], ps))
        states = list(map(lambda x: x[1], ps))
        idx    = np.random.choice(len(states), p=probs)

        return states[idx]
    return nextState

Para confirmar se entendeu esse bloco:
- Seja uma sequência-amostra:
`model = {(1, 1): (0.34401, 1), (1, 2): (0.42668, 1), (1, 3): (1.13914, 2),
(2, 3): (0.734, 1), (3, 3): (0.86, 1), (4, 3): (1.0, 1)})`  
O que representa cada elemento na variável `model`?  
Note que o Estado (1,3) acontece duas vezes.  

- Esta sequência-amostra gera a saída:

Direct Estimation:  
| 0.57 || 0.73 || 0.86 || 1.00 |  
| 0.43 || ------- || ------- || ------- |  
| 0.34 || ------- || ------- || ------- |  

Por que o valor do estado (1,3) é 0.57?

# Algoritmo Diferença-Temporal (TD)

O algoritmo TD usa a informação da Equação de Bellman para ajustar as estimativas de utilidade.


In [6]:
def td(percept, state, gamma, alpha):
    sn, rn         = percept
    s, a, r, pi, U = state

    if sn not in U:
        U[sn] = rn
    if s != (0,0):
        U[s]  = U[s] + alpha*(r + gamma*U[sn] - U[s])
    return pi[sn], (sn, pi[sn], rn, pi, U)

def runTD(mdp, pi, nTrials, alpha):
    S, A, R, P, gamma = mdp
    U         = {}
    nextState = performAction(pi, P)
    s0        = (1,1)
    goals     = [(4,3), (4,2)]

    for t in range(nTrials):
        # Estado inicial nulo
        state = (0,0), "", 0, pi, U
        s = s0
        percept = (s, R[s])
        g       = gamma
        while state[0] not in goals:
            a, state = td(percept, state, g, alpha)
            s        = nextState(s)
            percept  = (s, R[s])

    s, a, r, pi, U = state
    return U


Para confirmar se entendeu esse bloco:  
- Qual o motivo do uso do estado nulo `( state = (0,0), "", 0, pi, U )`?  
- O seguinte resultado foi obtido usando-se 1 trial e `alpha = 1`:  

| -0.08 || -0.08 || 0.86 || 1.00 |  
| -0.08 || -------- || ------- || ------- |  
| -0.08 || -------- || ------- || ------- |  

Algum estado repetiu? Por que foram observados esses valores?

# Comparação dos Algoritmos

Função principal.  
Note que a política está sendo estimada pelo método de Iteração-de-Política.

In [7]:
def main():
    mdp = createMDP()
    # Politica sendo estimada por:
    pi  = policyIteration(mdp)

    model = runDirectEst(mdp, pi, 1000)

    print("Direct Estimation: \n")
    for j in range(3,0,-1):
        for i in range(1,5):
            if (i,j) in model:
                soma, n = model[(i,j)]
                v       = soma / n
                print(f"|  {v:.2f}  |", end="")
            else:
                print("|       |", end="")
        print("")

    U = runTD(mdp, pi, 1000, 0.1)

    print("Time Difference: \n")
    for j in range(3,0,-1):
        for i in range(1,5):
            if (i,j) in U:
                v = U[(i,j)]
                print(f"|  {v:.2f}  |", end="")
            else:
                print("|       |", end="")
        print("")

main()

Direct Estimation: 

|  0.51  ||  0.65  ||  0.79  ||  1.00  |
|  0.39  ||       ||  0.52  ||  -1.00  |
|  0.29  ||  0.22  ||  0.28  ||  -0.78  |
Time Difference: 

|  0.51  ||  0.63  ||  0.81  ||  1.00  |
|  0.41  ||       ||  0.47  ||  -1.00  |
|  0.30  ||  0.18  ||  0.28  ||  -0.56  |
